# 10年定着予測 - reference/ノートブック由来の未実装アイデア（59_）

## 位置づけ

`reference/`ディレクトリの2本のノートブックを解析した結果、自分の441列パイプライン（`54_` R0_memofix_plus_LM、
現在の単層CatBoost最良・Public 0.515030）に対して**未実装だった2つの技法**を移植・検証する。

- `analysis_0812.ipynb`（Public 0.5568419345849898、自分のスコアより劣る）から:
  1. **自己学習（詳細）列のパース特徴量（SLブロック）** — [[eda-v6-findings]]で「有望・未使用」とメモしていた列の実装
  2. **Plain/Ordered CatBoostブレンド + 時系列2段Platt較正** — アルゴリズムが構造的に異なる2つのboosting_typeを
     ブレンドする、シード平均とは別種の多様性の軸
- `長期定着予測_crmaine_0813.ipynb` — セル0に「SIGNATE『長期定着人材の予測』の**序盤**に進めた技法を、
  他者への展開を考慮して丁寧に整理するもの」と明記されており、Publicスコアの記載もない。教材用の入門ノートブック
  であり、実装内容（TF-IDF+SVD10次元のみ・メモパーサー未修正・早期退職者除外なし）は自分の441列パイプラインより
  素朴だったため、**新規に移植すべき技法は無かった**。

## reference側にあったが移植しなかったもの（判断理由）

- `INTERACTION_PAIRS`（部署ID__first×__last等の生カテゴリ結合文字列）:
  自分のパイプラインには既に `部署ID_changes` / `上司ID_changes` / `勤務地_changes`（変化回数）や
  `_diff` / `_ratio`（最終月/初月）が同等の情報として入っている。[[kitchen-sink-combination-search]]で
  546列の総当たり探索をやっても新規の独立した特徴量が0件だった結論、および[[gbdt-interaction-type-matters]]
  の「GBDTは生カテゴリのANDを自力で学習できる」という知見を踏まえ、優先度が低いと判断し見送った。
- early6/late6/delta（0-5ヶ月 vs 18-23ヶ月の平均差）: 自分のパイプラインの`_early_mean`/`_late_mean`/
  `_late_minus_early`/`_late_early_ratio`（16指標）が同種の情報をより広いカバレッジで既に提供している。
- 360度評価列の欠損率: `create_missing_value_features`で既に実装済み。

## 検証設計（1変数ずつ分離）

| 行 | 特徴量 | モデル | 何を測るか |
|---|---|---|---|
| Row1 | R0_memofix_plus_LM（444列、`54_`と同一） | 標準パイプライン（Plain・較正なし・8シード平均） | ベースライン（`54_`の再現） |
| Row2 | R0_memofix_plus_LM + SL（自己学習パース追加） | 標準パイプライン（Row1と同一） | SLブロックの効果（Row2-Row1） |
| Row3 | R0_memofix_plus_LM（444列、Row1と同一特徴量） | Plain/Orderedブレンド + Platt較正 | モデリング手法の効果（Row3-Row1、特徴量は固定） |

採否は[[validation-asymmetry]]の通りPublicのみで判断する。検証は足切り（`VAL_REJECT_MARGIN`超の悪化）専用。


In [1]:
!pip install -q catboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 23.8 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.1 MB/s eta 0:00:00


In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [5]:
SCRIPT_NAME = "59_reference_ideas"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-15 11:07:53] [INFO] === [59_reference_ideas] 実験開始 ===


INFO:59_reference_ideas:=== [59_reference_ideas] 実験開始 ===


[2026-08-15 11:07:58] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


INFO:59_reference_ideas:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260815


[2026-08-15 11:07:58] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/59_reference_ideas_checkpoint.csv


INFO:59_reference_ideas:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/59_reference_ideas_checkpoint.csv


[2026-08-15 11:07:58] [INFO] チェックポイントは未作成（新規実行）


INFO:59_reference_ideas:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


[2026-08-15 11:08:03] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:59_reference_ideas:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-15 11:08:03] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:59_reference_ideas:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-15 11:08:03] [INFO] 定着率: 0.5647


INFO:59_reference_ideas:定着率: 0.5647


[2026-08-15 11:08:03] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:59_reference_ideas:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-15 11:08:03] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:59_reference_ideas:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-15 11:08:03] [INFO] Test  早期退職者: 0名 / 2502名


INFO:59_reference_ideas:Test  早期退職者: 0名 / 2502名


[2026-08-15 11:08:03] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:59_reference_ideas:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-15 11:08:03] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:59_reference_ideas:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`54_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-15 11:08:03] [INFO] ------------------------------------------------------------


INFO:59_reference_ideas:------------------------------------------------------------


[2026-08-15 11:08:03] [INFO] split非依存の基本特徴量を生成中...


INFO:59_reference_ideas:split非依存の基本特徴量を生成中...


[2026-08-15 11:08:03] [INFO] ------------------------------------------------------------


INFO:59_reference_ideas:------------------------------------------------------------


[2026-08-15 11:13:47] [INFO] split非依存の基本特徴量生成完了


INFO:59_reference_ideas:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`54_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-15 11:13:47] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:59_reference_ideas:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-15 11:13:49] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:59_reference_ideas:入社時メモ: SVD累積寄与率=0.760


[2026-08-15 11:13:53] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:59_reference_ideas:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-15 11:13:54] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:59_reference_ideas:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-15 11:13:54] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:59_reference_ideas:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`54_`と同一・継続採用）

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-15 11:13:54] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:59_reference_ideas:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-15 11:15:58] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:59_reference_ideas:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`54_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-15 11:15:58] [INFO] Persona単位の基本特徴量を生成中...


INFO:59_reference_ideas:Persona単位の基本特徴量を生成中...


[2026-08-15 11:15:58] [INFO] Persona単位の基本特徴量処理完了


INFO:59_reference_ideas:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、`49_`でパーサーを修正・`54_`と同一）

In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 49_: 見出しがない書式B（276件、5.24%）のフォールバック。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-15 11:15:58] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:59_reference_ideas:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-15 11:15:58] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:59_reference_ideas:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-15 11:15:58] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:59_reference_ideas:L_v2: Train (2761, 3), Test (2502, 3)


## 6. L2×Mリスク要因数（`54_`で確認済み、Public -0.004119・そのまま採用）

In [14]:
_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)

    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)

    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info(f"L2xMリスク特徴量: Train {train_l2m.shape}, Test {test_l2m.shape}")
print(train_l2m["L2xM_リスク要因数"].value_counts().sort_index())


[2026-08-15 11:15:58] [INFO] L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


INFO:59_reference_ideas:L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


L2xM_リスク要因数
0    2025
1     689
2      47
Name: count, dtype: int64


## 7. 自己学習（詳細）パース特徴量（SLブロック、`59_`で新規追加）

`reference/analysis_0812.ipynb` の `make_learning_features()` を移植。`自己学習（詳細）` 列は
「コース名：X時間｜コース名：Y時間...」形式のテキストで、[[eda-v6-findings]]で「有望・未使用」と
メモしていたが実装していなかった。今回、コースイベント数・ユニークコース数・総学習時間に加え、
コース名のキーワードでDX_IT/対人営業/管理リスク/組織業務の4トピックに分類した時間合計を追加する。

leak安全性: `train_monthly`/`test_monthly` は既に0-23ヶ月に絞り込まれているため、追加のフィルタは不要
（`54_`の他ブロックと同じ前提）。


In [15]:
def create_self_study_features(monthly_df, employee_ids):
    """自己学習（詳細）列を｜区切りでパースし、コースイベント数・総時間・トピック別時間を集計する
    （reference/analysis_0812.ipynb の make_learning_features を移植）。
    """
    z = monthly_df[monthly_df[ID_COL].isin(employee_ids)][[ID_COL, "経過月数", "自己学習（詳細）"]].copy()
    z = z[z["自己学習（詳細）"].notna() & z["自己学習（詳細）"].ne("受講なし")]

    out = pd.DataFrame(index=pd.Index(employee_ids, name=ID_COL))
    out["学習_active_months"] = z.groupby(ID_COL)["経過月数"].nunique()

    tokens = z.assign(token=z["自己学習（詳細）"].astype(str).str.split("｜")).explode("token")
    extracted = tokens["token"].str.extract(r"^(.*)：([0-9.]+)時間$")
    tokens["course"] = extracted[0]
    tokens["hours"] = pd.to_numeric(extracted[1], errors="coerce")
    tokens = tokens.dropna(subset=["course", "hours"])

    out["学習_course_events"] = tokens.groupby(ID_COL).size()
    out["学習_unique_courses"] = tokens.groupby(ID_COL)["course"].nunique()
    out["学習_total_hours_parsed"] = tokens.groupby(ID_COL)["hours"].sum()

    course_buckets = {
        "DX_IT": ["Python", "SQL", "クラウド", "データ", "システム", "RPA", "AI", "統計", "アジャイル"],
        "対人営業": ["顧客", "提案", "交渉", "CRM", "アカウント", "ファシリ", "コミュニケーション"],
        "管理リスク": ["リスク", "コンプライアンス", "内部統制", "労務", "管理会計", "品質管理"],
        "組織業務": ["組織", "人材", "業務", "プロセス", "プロジェクト"],
    }
    course_text = tokens["course"].astype(str)
    for bucket_name, words in course_buckets.items():
        mask = course_text.apply(lambda s: any(word in s for word in words))
        out[f"学習時間_{bucket_name}"] = tokens.loc[mask].groupby(ID_COL)["hours"].sum()

    return out.fillna(0.0).reset_index()


logger.info("自己学習（詳細）パース特徴量(SL)を生成中...")
train_sl = create_self_study_features(train_monthly, train_ids)
test_sl = create_self_study_features(test_monthly, test_ids)
logger.info(f"SL: Train {train_sl.shape}, Test {test_sl.shape}")
print(train_sl.describe().round(2))


[2026-08-15 11:15:58] [INFO] 自己学習（詳細）パース特徴量(SL)を生成中...


INFO:59_reference_ideas:自己学習（詳細）パース特徴量(SL)を生成中...


[2026-08-15 11:15:59] [INFO] SL: Train (2761, 9), Test (2502, 9)


INFO:59_reference_ideas:SL: Train (2761, 9), Test (2502, 9)


       学習_active_months  学習_course_events  学習_unique_courses  \
count           2761.00           2761.00            2761.00   
mean               9.34             14.65               9.07   
std                3.33              7.39               4.75   
min                0.00              0.00               0.00   
25%                7.00              9.00               5.00   
50%                9.00             13.00               8.00   
75%               12.00             19.00              12.00   
max               22.00             40.00              25.00   

       学習_total_hours_parsed  学習時間_DX_IT  学習時間_対人営業  学習時間_管理リスク  学習時間_組織業務  
count                2761.00     2761.00    2761.00     2761.00    2761.00  
mean                   40.92       13.11       6.80        8.22       4.60  
std                    17.97       11.69      10.17        8.35       5.83  
min                     0.00        0.00       0.00        0.00       0.00  
25%                    28.00        4.

## 8. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`54_`と同一ロジックに、`extra_blocks`へ`"SL"`を追加した場合の分岐だけを足す。


In [16]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（54_と同一 + SLブロック対応）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        tf = tf.merge(train_l2m, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_l2m, on=ID_COL, how="left")

    if "SL" in extra_blocks:
        tf = tf.merge(train_sl, on=ID_COL, how="left")
        ttf = ttf.merge(test_sl, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（SLブロック対応版）")


✅ 部署Target Encoding・prepare_split関数定義完了（SLブロック対応版）


## 9. 特徴量の組み立て（extra_blocks={"L2","SL"}を常時マージし、CONFIGSで列選択を切替）

In [17]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2", "SL"}

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-15 11:15:59] [INFO] ============================================================


INFO:59_reference_ideas:============================================================


[2026-08-15 11:15:59] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:59_reference_ideas:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-15 11:15:59] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:59_reference_ideas:main_train=2208, main_valid(生存者)=535


[2026-08-15 11:16:00] [INFO] 全件=2761


INFO:59_reference_ideas:全件=2761


[2026-08-15 11:16:00] [INFO] 特徴量数: 452


INFO:59_reference_ideas:特徴量数: 452


## 10. 特徴量グループの棚卸し（SLグループを追加）

In [18]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "LM":        _cols_of(train_l2m),
    "SL":        _cols_of(train_sl),
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 452 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  SL            8 列   例: ['学習_active_months', '学習_course_events']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  LM            3 列   例: ['M_不適合', 'L2xM_ダブル不適合']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']

## 11. 構成の事前登録

`54_`のA_PARAMS/反復数をそのまま流用する（ハイパラ探索はしない＝「特徴量・モデリングだけの差」にするため）。


In [19]:
A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}

ITER_HOLDOUT = 560   # 38_・54_と同一
ITER_FULL    = 560

SEEDS_SUB = [42, 2024, 7, 1234, 99]
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]

CONFIGS = {
    "R0_memofix_plus_LM":    {"groups": ALL_GROUPS - {"SL"}},   # 54_のベスト構成の再現（444列）
    "R0_plus_LM_plus_SL":    {"groups": ALL_GROUPS},            # + 自己学習パース特徴量
}

VAL_REJECT_MARGIN = 0.02


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


print(f"{'config':<24s} {'列数':>5s}")
print("-" * 40)
for name, spec in CONFIGS.items():
    print(f"{name:<24s} {len(cols_for(spec, ag_train_80b)):>5d}")


config                      列数
----------------------------------------
R0_memofix_plus_LM         444
R0_plus_LM_plus_SL         452


## 12. モデル関数（標準パイプライン、`54_`と同一。反復数固定・early stoppingなし）

In [20]:
def _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed):
    model = cb.CatBoostClassifier(
        **params, iterations=int(n_iter), random_seed=seed,
        verbose=False, cat_features=obj_cols, task_type="CPU",
    )
    model.fit(X_tr, y_tr)
    return model


def fit_holdout_fixed(ag_train, ag_val, feature_cols, params, n_iter, seeds):
    obj_cols = [c for c in feature_cols if ag_train[c].dtype == "object"]
    X_tr, y_tr = ag_train[feature_cols].fillna(-999), ag_train[TARGET_COL]
    X_va, y_va = ag_val[feature_cols].fillna(-999), ag_val[TARGET_COL]

    val_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        val_preds.append(model.predict_proba(X_va)[:, 1])
    val_preds = np.array(val_preds)

    singles = [log_loss(y_va, vp) for vp in val_preds]
    return {
        "val_seedavg": float(log_loss(y_va, val_preds.mean(axis=0))),
        "val_single_mean": float(np.mean(singles)),
        "val_single_sd": float(np.std(singles)),
        "val_preds": val_preds,
        "y_val": y_va.values,
    }


def fit_full_fixed(ag_full, test_feats, feature_cols, params, n_iter, seeds):
    obj_cols = [c for c in feature_cols if ag_full[c].dtype == "object"]
    X_tr, y_tr = ag_full[feature_cols].fillna(-999), ag_full[TARGET_COL]
    X_test = test_feats[feature_cols].fillna(-999)

    test_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, params, n_iter, seed)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"    seed={seed}: 全件学習完了")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)

print("✅ モデル関数定義完了（fit_holdout_fixed / fit_full_fixed）")


✅ モデル関数定義完了（fit_holdout_fixed / fit_full_fixed）


## 13. チェックポイント

In [21]:
RESULT_SCHEMA = [
    "config", "kind", "n_features",
    "val_seedavg", "val_single_mean", "val_single_sd",
    "val_plain_raw", "val_plain_cal", "val_ordered_raw", "val_ordered_cal", "val_hybrid_cal",
    "n_iterations", "n_train", "pred_mean", "submission_path",
]


def make_row(**kwargs):
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)


def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)


def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label] if len(checkpoint) else checkpoint
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")


✅ チェックポイント関数定義完了


## 14. Row1・Row2: 標準パイプラインでベースライン・SL構成を実行

In [22]:
def make_standard_runner(config_label, spec):
    def _run():
        feats = cols_for(spec, ag_train_80b)
        feats_full = cols_for(spec, ag_full)
        assert feats == feats_full, "検証と全件学習で特徴量列が食い違っている"

        logger.info("=" * 60)
        logger.info(f"[{config_label}] {len(feats)}列")

        hold = fit_holdout_fixed(ag_train_80b, ag_val_surv, feats, A_PARAMS, ITER_HOLDOUT, SEEDS_VAL)
        logger.info(f"  検証(生存者{len(ag_val_surv)}名): シード平均 {hold['val_seedavg']:.6f} "
                    f"/ 単一シード {hold['val_single_mean']:.6f} ± {hold['val_single_sd']:.6f}")
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy", hold["val_preds"])

        test_preds = fit_full_fixed(ag_full, test_features_full, feats, A_PARAMS, ITER_FULL, SEEDS_SUB)
        preds = test_preds.mean(axis=0)
        path = save_submission(test_features_full.index, preds, config_label)

        return make_row(
            config=config_label, kind="standard", n_features=len(feats),
            val_seedavg=hold["val_seedavg"], val_single_mean=hold["val_single_mean"],
            val_single_sd=hold["val_single_sd"],
            n_iterations=ITER_FULL, n_train=len(ag_full), pred_mean=float(preds.mean()),
            submission_path=path,
        )
    return _run


standard_results = {}
for _name, _spec in CONFIGS.items():
    standard_results[_name] = run_or_resume(_name, make_standard_runner(_name, _spec))

print()
print(f"{'config':<24s} {'列数':>5s} {'val(8シード平均)':>16s} {'単一sd':>9s}")
print("-" * 58)
for _name, _r in standard_results.items():
    print(f"{_name:<24s} {int(_r['n_features']):>5d} {float(_r['val_seedavg']):>16.6f} {float(_r['val_single_sd']):>9.6f}")


[2026-08-15 11:16:00] [INFO] ============================================================


INFO:59_reference_ideas:============================================================


[2026-08-15 11:16:00] [INFO] [R0_memofix_plus_LM] 444列


INFO:59_reference_ideas:[R0_memofix_plus_LM] 444列


[2026-08-15 11:16:40] [INFO]   検証(生存者535名): シード平均 0.505477 / 単一シード 0.508987 ± 0.005181


...

[2026-08-15 11:18:19] [INFO]     seed=99: 全件学習完了


INFO:59_reference_ideas:    seed=99: 全件学習完了


[2026-08-15 11:18:19] [INFO]   提出ファイル: 20260815_59_reference_ideas_R0_plus_LM_plus_SL.csv（予測平均=0.5914）


INFO:59_reference_ideas:  提出ファイル: 20260815_59_reference_ideas_R0_plus_LM_plus_SL.csv（予測平均=0.5914）



config                      列数      val(8シード平均)      単一sd
----------------------------------------------------------
R0_memofix_plus_LM         444         0.505477  0.005181
R0_plus_LM_plus_SL         452         0.493946  0.004662


## 15. Row3: Plain/Ordered CatBoostブレンド + Platt較正（`reference/analysis_0812.ipynb`の移植）

特徴量は `R0_memofix_plus_LM`（Row1と完全に同一の444列）に固定し、**モデリング手法だけ**を変える。

- `boosting_type="Plain"`（3シード平均）と `boosting_type="Ordered"`（1シード、計算コストが高いため）を50:50でブレンド
- 較正: `main_train`（`ag_train_80b`）を入社日でさらに8:2に割り、前半`cal_train`で学習した予測を後半`cal_valid`で
  Platt較正（ロジスティック回帰）する。この較正器を検証（`main_valid`=`ag_val_surv`）とTest予測の両方に適用する

**設計上の妥協点（reference notebookと同じ）**: 提出用の全件学習モデルは`cal_valid`を含む`ag_full`全体で
学習するため、較正器と最終モデルの学習データに軽微な重複がある。較正曲線はロジスティック回帰1本の単純な
ものなので実務上の影響は小さいと考えられるが、移植元の設計をそのまま踏襲した結果でありbias-freeではない。


In [23]:
PLAIN_SEEDS = [42, 2024, 7]
ORDERED_SEED = 42
BLEND_WEIGHT_PLAIN = 0.50


def _to_logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _fit_platt(raw_prob, y_true):
    lr = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2000)
    lr.fit(_to_logit(raw_prob).reshape(-1, 1), np.asarray(y_true, dtype=int))
    return lr


def _apply_platt(calibrator, raw_prob):
    return calibrator.predict_proba(_to_logit(raw_prob).reshape(-1, 1))[:, 1]


def run_plain_ordered_blend(feature_cols, params, n_iter_holdout, n_iter_full):
    obj_cols = [c for c in feature_cols if ag_train_80b[c].dtype == "object"]

    hire_date = train_persona.set_index(ID_COL)["入社日"]
    main_train_sorted = ag_train_80b.assign(_入社日=hire_date.loc[ag_train_80b.index]).sort_values("_入社日")
    cal_split = int(len(main_train_sorted) * 0.8)
    cal_train = main_train_sorted.iloc[:cal_split]
    cal_valid = main_train_sorted.iloc[cal_split:]
    logger.info(f"  cal_train={len(cal_train)}, cal_valid={len(cal_valid)}（main_trainを入社日でさらに8:2分割）")

    def _xy(df):
        return df[feature_cols].fillna(-999), df[TARGET_COL]

    # --- cal段階: 較正器を学習 ---
    X_ct, y_ct = _xy(cal_train)
    X_cv, y_cv = _xy(cal_valid)

    plain_cal_raw_parts = []
    for seed in PLAIN_SEEDS:
        m = cb.CatBoostClassifier(**params, boosting_type="Plain", iterations=n_iter_holdout,
                                   random_seed=seed, verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_ct, y_ct)
        plain_cal_raw_parts.append(m.predict_proba(X_cv)[:, 1])
    plain_cal_raw = np.mean(plain_cal_raw_parts, axis=0)
    plain_calibrator = _fit_platt(plain_cal_raw, y_cv)

    ordered_cal_model = cb.CatBoostClassifier(**params, boosting_type="Ordered", iterations=n_iter_holdout,
                                               random_seed=ORDERED_SEED, verbose=False, cat_features=obj_cols, task_type="CPU")
    ordered_cal_model.fit(X_ct, y_ct)
    ordered_cal_raw = ordered_cal_model.predict_proba(X_cv)[:, 1]
    ordered_calibrator = _fit_platt(ordered_cal_raw, y_cv)

    logger.info(f"  [cal] Plain raw logloss={log_loss(y_cv, plain_cal_raw):.6f} / "
                f"Ordered raw logloss={log_loss(y_cv, ordered_cal_raw):.6f}")

    # --- main段階: main_train全体で学習し、main_valid(生存者)で検証 ---
    X_mt, y_mt = _xy(ag_train_80b)
    X_mv, y_mv = _xy(ag_val_surv)

    plain_val_parts = []
    for seed in PLAIN_SEEDS:
        m = cb.CatBoostClassifier(**params, boosting_type="Plain", iterations=n_iter_holdout,
                                   random_seed=seed, verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_mt, y_mt)
        plain_val_parts.append(m.predict_proba(X_mv)[:, 1])
    plain_val_raw = np.mean(plain_val_parts, axis=0)
    plain_val_cal = _apply_platt(plain_calibrator, plain_val_raw)

    ordered_val_model = cb.CatBoostClassifier(**params, boosting_type="Ordered", iterations=n_iter_holdout,
                                               random_seed=ORDERED_SEED, verbose=False, cat_features=obj_cols, task_type="CPU")
    ordered_val_model.fit(X_mt, y_mt)
    ordered_val_raw = ordered_val_model.predict_proba(X_mv)[:, 1]
    ordered_val_cal = _apply_platt(ordered_calibrator, ordered_val_raw)

    hybrid_val = BLEND_WEIGHT_PLAIN * plain_val_cal + (1.0 - BLEND_WEIGHT_PLAIN) * ordered_val_cal

    val_scores = {
        "val_plain_raw": log_loss(y_mv, plain_val_raw),
        "val_plain_cal": log_loss(y_mv, plain_val_cal),
        "val_ordered_raw": log_loss(y_mv, ordered_val_raw),
        "val_ordered_cal": log_loss(y_mv, ordered_val_cal),
        "val_hybrid_cal": log_loss(y_mv, hybrid_val),
    }
    logger.info(f"  検証(生存者{len(ag_val_surv)}名): "
                f"Plain較正後={val_scores['val_plain_cal']:.6f} / "
                f"Ordered較正後={val_scores['val_ordered_cal']:.6f} / "
                f"Hybrid50:50={val_scores['val_hybrid_cal']:.6f}")

    # --- 提出用: 全件(ag_full)で学習、cal段階の較正器を流用 ---
    X_full, y_full = _xy(ag_full)
    X_test = test_features_full[feature_cols].fillna(-999)

    plain_test_parts = []
    for seed in PLAIN_SEEDS:
        m = cb.CatBoostClassifier(**params, boosting_type="Plain", iterations=n_iter_full,
                                   random_seed=seed, verbose=False, cat_features=obj_cols, task_type="CPU")
        m.fit(X_full, y_full)
        plain_test_parts.append(m.predict_proba(X_test)[:, 1])
        logger.info(f"    Plain seed={seed}: 全件学習完了")
    plain_test_raw = np.mean(plain_test_parts, axis=0)
    plain_test_cal = _apply_platt(plain_calibrator, plain_test_raw)

    ordered_test_model = cb.CatBoostClassifier(**params, boosting_type="Ordered", iterations=n_iter_full,
                                                random_seed=ORDERED_SEED, verbose=False, cat_features=obj_cols, task_type="CPU")
    ordered_test_model.fit(X_full, y_full)
    logger.info("    Ordered: 全件学習完了")
    ordered_test_raw = ordered_test_model.predict_proba(X_test)[:, 1]
    ordered_test_cal = _apply_platt(ordered_calibrator, ordered_test_raw)

    test_pred = BLEND_WEIGHT_PLAIN * plain_test_cal + (1.0 - BLEND_WEIGHT_PLAIN) * ordered_test_cal

    return val_scores, test_pred

print("✅ Plain/Orderedブレンド + Platt較正関数定義完了")


✅ Plain/Orderedブレンド + Platt較正関数定義完了


In [24]:
def _run_blend():
    config_label = "R0_memofix_plus_LM_PlainOrderedCal"
    feats = cols_for(CONFIGS["R0_memofix_plus_LM"], ag_train_80b)
    logger.info("=" * 60)
    logger.info(f"[{config_label}] {len(feats)}列（Row1と同一特徴量、モデリングのみ変更）")

    val_scores, test_pred = run_plain_ordered_blend(feats, A_PARAMS, ITER_HOLDOUT, ITER_FULL)
    path = save_submission(test_features_full.index, test_pred, config_label)

    return make_row(
        config=config_label, kind="plain_ordered_blend", n_features=len(feats),
        val_plain_raw=val_scores["val_plain_raw"], val_plain_cal=val_scores["val_plain_cal"],
        val_ordered_raw=val_scores["val_ordered_raw"], val_ordered_cal=val_scores["val_ordered_cal"],
        val_hybrid_cal=val_scores["val_hybrid_cal"],
        n_iterations=ITER_FULL, n_train=len(ag_full), pred_mean=float(test_pred.mean()),
        submission_path=path,
    )


blend_result = run_or_resume("R0_memofix_plus_LM_PlainOrderedCal", _run_blend)
print()
print("Plain/Orderedブレンド結果:")
for k in ["val_plain_raw", "val_plain_cal", "val_ordered_raw", "val_ordered_cal", "val_hybrid_cal"]:
    print(f"  {k:<16s} {float(blend_result[k]):.6f}")


[2026-08-15 11:18:19] [INFO] ============================================================


INFO:59_reference_ideas:============================================================


[2026-08-15 11:18:19] [INFO] [R0_memofix_plus_LM_PlainOrderedCal] 444列（Row1と同一特徴量、モデリングのみ変更）


INFO:59_reference_ideas:[R0_memofix_plus_LM_PlainOrderedCal] 444列（Row1と同一特徴量、モデリングのみ変更）


[2026-08-15 11:18:19] [INFO]   cal_train=1766, cal_valid=442（main_trainを入社日でさらに8:2分割）


...

[2026-08-15 11:19:42] [INFO]     Ordered: 全件学習完了


INFO:59_reference_ideas:    Ordered: 全件学習完了


[2026-08-15 11:19:42] [INFO]   提出ファイル: 20260815_59_reference_ideas_R0_memofix_plus_LM_PlainOrderedCal.csv（予測平均=0.5948）


INFO:59_reference_ideas:  提出ファイル: 20260815_59_reference_ideas_R0_memofix_plus_LM_PlainOrderedCal.csv（予測平均=0.5948）



Plain/Orderedブレンド結果:
  val_plain_raw    0.503745
  val_plain_cal    0.507326
  val_ordered_raw  0.510709
  val_ordered_cal  0.513966
  val_hybrid_cal   0.508535


## 16. 結果まとめ

In [25]:
baseline_val = float(standard_results["R0_memofix_plus_LM"]["val_seedavg"])
sl_val = float(standard_results["R0_plus_LM_plus_SL"]["val_seedavg"])
hybrid_val = float(blend_result["val_hybrid_cal"])

print("=" * 70)
print("Row1 baseline (R0_memofix_plus_LM, 標準パイプライン)      :", f"{baseline_val:.6f}")
print("Row2 + SLブロック (標準パイプライン)                       :", f"{sl_val:.6f}",
      f"  差={sl_val - baseline_val:+.6f}")
print("Row3 Plain/Orderedブレンド+較正（Row1と同一特徴量）        :", f"{hybrid_val:.6f}",
      f"  差={hybrid_val - baseline_val:+.6f}")
print("=" * 70)

rows = []
for name, r in standard_results.items():
    rows.append({"config": name, "列数": int(r["n_features"]), "val": float(r["val_seedavg"]),
                 "ファイル": Path(r["submission_path"]).name})
rows.append({"config": blend_result["config"], "列数": int(blend_result["n_features"]),
             "val": hybrid_val, "ファイル": Path(blend_result["submission_path"]).name})
summary = pd.DataFrame(rows).set_index("config")

summary["提出"] = "提出する"
for name, val in [("R0_plus_LM_plus_SL", sl_val), ("R0_memofix_plus_LM_PlainOrderedCal", hybrid_val)]:
    if val - baseline_val > VAL_REJECT_MARGIN:
        summary.loc[name, "提出"] = "見送り（足切り）"

pd.set_option("display.width", 200)
print()
print(summary.to_string())
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv")
logger.info(f"サマリを保存: {TODAY}_{SCRIPT_NAME}_summary.csv")


Row1 baseline (R0_memofix_plus_LM, 標準パイプライン)      : 0.505477
Row2 + SLブロック (標準パイプライン)                       : 0.493946   差=-0.011531
Row3 Plain/Orderedブレンド+較正（Row1と同一特徴量）        : 0.508535   差=+0.003058

                                     列数       val                                                                ファイル    提出
config                                                                                                                     
R0_memofix_plus_LM                  444  0.505477                  20260815_59_reference_ideas_R0_memofix_plus_LM.csv  提出する
R0_plus_LM_plus_SL                  452  0.493946                  20260815_59_reference_ideas_R0_plus_LM_plus_SL.csv  提出する
R0_memofix_plus_LM_PlainOrderedCal  444  0.508535  20260815_59_reference_ideas_R0_memofix_plus_LM_PlainOrderedCal.csv  提出する
[2026-08-15 11:19:42] [INFO] サマリを保存: 20260815_59_reference_ideas_summary.csv


INFO:59_reference_ideas:サマリを保存: 20260815_59_reference_ideas_summary.csv


## 17. 提出方針

### 判定（事前登録・[[validation-asymmetry]]と同様の考え方）

- **採否は Public のみ。** 検証は足切り（悪化検出）専用。
- **足切り**: `VAL_REJECT_MARGIN`（0.02）を超えて悪化したら提出しない。
- Row2（SL）とRow3（Plain/Orderedブレンド+較正）は、Row1という同一のベースラインに対する
  **単一の事前登録済み介入**なので、両方提出してよい（複数構成から検証スコアで選ぶ探索ではない）。

### 期待値について

- SLブロック: [[eda-v6-findings]]で「有望・未使用」だった列の初実装。効くかどうかは未知数。
- Plain/Orderedブレンド+較正: シード平均（[[modeling-levers-beat-new-features]]で「保険であって
  精度向上ではない」と確認済み）とは異なり、boosting_typeという構造的に異なる軸のブレンドなので、
  真の多様性を生む可能性がある。ただし較正器の軽微なリーク（上記15節）があるため、Publicでの
  結果は額面通り受け取らず、[[validation-asymmetry]]の通りPublicでの実測を待って判断する。
